In [ ]:
import numpy as np
import pandas as pd


df = pd.read_csv('Forza_Horizon_Cars_Tratado.csv') 


atributos = ['Stock_Rating', '0-100_Mph']


estatisticas_modelos = {}

total_carros = len(df)


for tipo_modelo in df['Model_type'].unique():
    

    carros_desse_tipo = df[df['Model_type'] == tipo_modelo]
    

    priori = len(carros_desse_tipo) / total_carros
    

    medias = carros_desse_tipo[atributos].mean()
    desvios = carros_desse_tipo[atributos].std()
    

    estatisticas_modelos[tipo_modelo] = {
        'priori': priori,
        'medias': medias,
        'desvios': desvios
    }

print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


In [ ]:
def probabilidade_gaussiana(x, media, desvio_padrao):

    if desvio_padrao == 0:
        desvio_padrao = 1e-6
        
    termo_1 = 1 / (np.sqrt(2 * np.pi) * desvio_padrao)
    exponente = np.exp(-((x - media)**2 / (2 * desvio_padrao**2)))
    
    return termo_1 * exponente

In [ ]:
def prever_tipo_de_carro(stock_rating_novo, tempo_novo):
    melhor_modelo = None
    maior_probabilidade = -1
    

    for modelo, stats in estatisticas_modelos.items():
        
        prob_priori = stats['priori']
        

        prob_stock = probabilidade_gaussiana(stock_rating_novo, 
                                             stats['medias']['Stock_Rating'], 
                                             stats['desvios']['Stock_Rating'])
        
        prob_tempo = probabilidade_gaussiana(tempo_novo, 
                                             stats['medias']['0-100_Mph'], 
                                             stats['desvios']['0-100_Mph'])
        

        probabilidade_posteriori = prob_priori * prob_stock * prob_tempo
        

        if probabilidade_posteriori > maior_probabilidade:
            maior_probabilidade = probabilidade_posteriori
            melhor_modelo = modelo
            
    return melhor_modelo


chute_do_algoritmo = prever_tipo_de_carro(950, 2.5)
print(f"O algoritmo de Bayes prevê que este carro é um: {chute_do_algoritmo}")

O algoritmo de Bayes prevê que este carro é um: RALLY MONSTERS


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

df_limpo = df.dropna(subset=['Stock_Rating', '0-100_Mph', 'Model_type'])


X = df_limpo[['Stock_Rating', '0-100_Mph']]
y = df_limpo['Model_type']


X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)


X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)


rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_treino, y_treino)
rf_previsoes = rf_model.predict(X_teste)

print("==== RESULTADOS: RANDOM FOREST ====")
print(f"Acurácia Geral: {accuracy_score(y_teste, rf_previsoes):.2f}\n")

print(classification_report(y_teste, rf_previsoes, zero_division=0))



knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_treino, y_treino)
knn_previsoes = knn_model.predict(X_teste)

print("\n\n==== RESULTADOS: KNN ====")
print(f"Acurácia Geral: {accuracy_score(y_teste, knn_previsoes):.2f}\n")
print(classification_report(y_teste, knn_previsoes, zero_division=0))

==== RESULTADOS: RANDOM FOREST ====
Acurácia Geral: 0.14

                     precision    recall  f1-score   support

     CLASSIC MUSCLE       0.00      0.00      0.00         0
CLASSIC SPORTS CARS       0.00      0.00      0.00         1
         DRIFT CARS       0.00      0.00      0.00         0
 EXTREME TRACK TOYS       0.00      0.00      0.00         1
          HYPERCARS       0.00      0.00      0.00         1
      MODERN MUSCLE       0.00      0.00      0.00         1
       MODERN RALLY       0.00      0.00      0.00         0
 MODERN SPORTS CARS       0.00      0.00      0.00         1
   MODERN SUPERCARS       0.33      0.33      0.33         3
    PICK-UP & 4X4'S       0.00      0.00      0.00         0
     RALLY MONSTERS       0.67      0.67      0.67         3
      RARE CLASSICS       0.00      0.00      0.00         2
      RETRO SALOONS       1.00      1.00      1.00         1
  RETRO SPORTS CARS       0.00      0.00      0.00         3
    RETRO SUPERCARS       